# KoELECTRA 추출 모델 평가
**모델**: `yunjeong116/koelectra-extractor` (HF Hub 파인튜닝 모델)  
**목표**: 학습이 제대로 됐는지 정량·정성 평가

| 평가 항목 | 내용 |
|---|---|
| 1. 모델 로드 확인 | HF Hub에서 정상 로드되는지 |
| 2. 상식 점검 | 명백한 할 일 / 노이즈 문장 분류 확인 |
| 3. 정량 평가 | v2.1 라벨 데이터로 Precision / Recall / F1 |
| 4. 임계값 분석 | threshold 0.5 적절한지 확인 |
| 5. 실제 통신문 테스트 | 전체 가정통신문 입력 → 할 일 추출 |

**실행 방법**
1. `런타임` → `런타임 유형 변경` → **T4 GPU** 선택
2. 셀을 위에서부터 순서대로 실행

## 1. 라이브러리 설치

In [ ]:
!pip install -q transformers==4.44.2 torch scikit-learn

## 2. 모델 로드 (HF Hub)

In [ ]:
import torch
import re
from typing import Optional
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_ID = "yunjeong116/koelectra-extractor"
BINARY_THRESHOLD = 0.5
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"디바이스: {device}")
print(f"모델 로드 중: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)
model.to(device)
model.eval()

print("\n✅ 모델 로드 완료")
print(f"   파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print(f"   레이블 매핑: {model.config.id2label}")

## 3. 추론 함수 정의

In [ ]:
def predict_sentence(sentence: str) -> tuple[int, float]:
    """단일 문장 → (label, confidence). label 1=할 일, 0=노이즈"""
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128,
    ).to(device)
    with torch.no_grad():
        prob = torch.softmax(model(**inputs).logits, dim=-1)[0]
    confidence = float(prob[1].item())
    label = 1 if confidence >= BINARY_THRESHOLD else 0
    return label, round(confidence, 4)


def predict_batch(sentences: list[str], batch_size: int = 32) -> list[tuple[int, float]]:
    """배치 추론 — 대량 평가용"""
    results = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128,
        ).to(device)
        with torch.no_grad():
            probs = torch.softmax(model(**inputs).logits, dim=-1)[:, 1]
        for p in probs.cpu().tolist():
            results.append((1 if p >= BINARY_THRESHOLD else 0, round(p, 4)))
    return results

print("추론 함수 정의 완료")

## 4. 상식 점검 (Sanity Check)
명백한 할 일 / 노이즈 문장을 넣어서 모델이 기본적인 방향을 잡고 있는지 확인합니다.  
**모든 ✅ 가 맞아야** 학습이 제대로 된 것입니다.

In [ ]:
sanity_cases = [
    # (문장, 정답 label, 설명)
    # ── 할 일이어야 하는 것 (label=1) ───────────────────────────────────
    ("4월 30일까지 체험학습 동의서를 제출해주시기 바랍니다.",        1, "제출 요청"),
    ("준비물: 도시락, 물통, 실내화를 꼭 지참해 주세요.",             1, "준비물 지참"),
    ("급식비 79,980원을 5월 10일까지 납부해 주시기 바랍니다.",       1, "비용 납부"),
    ("학교종이 앱 설문에 5월 7일까지 응답해 주시기 바랍니다.",       1, "설문 응답"),
    ("인플루엔자 예방접종 후 접종 확인서를 제출해 주십시오.",         1, "건강·안전"),
    ("신청서를 작성하여 담임선생님께 제출해 주시기 바랍니다.",        1, "신청서 제출"),
    # ── 노이즈여야 하는 것 (label=0) ───────────────────────────────────
    ("학부모님 안녕하십니까?",                                        0, "인사말"),
    ("항상 학교 교육에 관심을 가져주셔서 감사합니다.",                0, "감사 인사"),
    ("2026년 3월 4일 서울갈산초등학교장",                            0, "발신자"),
    ("본 안내문은 학교 홈페이지에 게시됩니다.",                       0, "순수 공지"),
    ("교무실 2649-7232~3. 팩스 2651-9881",                           0, "연락처 메타"),
    ("자살예방상담전화 109 / 청소년상담전화 1388",                    0, "푸터 노이즈"),
]

print(f"{'문장':<50} {'정답':>4} {'예측':>4} {'확률':>8}  결과")
print("-" * 80)

correct = 0
for sent, gt, desc in sanity_cases:
    pred, conf = predict_sentence(sent)
    ok = pred == gt
    correct += ok
    tag = "✅" if ok else "❌"
    gt_tag  = "할일" if gt   == 1 else "노이즈"
    pred_tag = "할일" if pred == 1 else "노이즈"
    print(f"{sent[:48]:<50} {gt_tag:>4} {pred_tag:>4} {conf:>8.4f}  {tag} {desc}")

print(f"\n상식 점검: {correct}/{len(sanity_cases)} 정답")
if correct == len(sanity_cases):
    print("✅ 모델이 기본 방향을 제대로 학습했습니다.")
elif correct >= len(sanity_cases) * 0.8:
    print("⚠️  대부분 맞지만 일부 케이스를 틀렸습니다. 정량 평가로 상세 확인 필요.")
else:
    print("❌ 학습이 충분하지 않습니다. 재학습 또는 데이터 보강 필요.")

## 5. 정량 평가 — v2.1 라벨 데이터
`v2.1_notices_galsan.jsonl` 파일을 업로드하세요.

In [ ]:
from google.colab import files
import json

print("v2.1_notices_galsan.jsonl 파일을 업로드하세요.")
uploaded = files.upload()

fname = list(uploaded.keys())[0]
records = [json.loads(l) for l in uploaded[fname].decode("utf-8").splitlines() if l.strip()]

texts  = [r["text"] for r in records]
y_true = [int(r["is_todo"]) for r in records]

true_cnt  = sum(y_true)
false_cnt = len(y_true) - true_cnt
print(f"\n로드 완료: 총 {len(records)}개")
print(f"  is_todo=True : {true_cnt}개 ({true_cnt/len(y_true)*100:.1f}%)")
print(f"  is_todo=False: {false_cnt}개 ({false_cnt/len(y_true)*100:.1f}%)")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

print("배치 추론 중... (5,475개)")
results   = predict_batch(texts, batch_size=64)
y_pred    = [r[0] for r in results]
y_conf    = [r[1] for r in results]

print("\n" + "=" * 55)
print("정량 평가 결과")
print("=" * 55)
print(classification_report(
    y_true, y_pred,
    target_names=["노이즈(0)", "할 일(1)"],
    digits=4,
    zero_division=0,
))

cm = confusion_matrix(y_true, y_pred)
print("혼동 행렬 (행=정답, 열=예측)")
print(f"              예측:노이즈  예측:할일")
print(f"정답:노이즈     {cm[0][0]:6d}      {cm[0][1]:5d}")
print(f"정답:할 일      {cm[1][0]:6d}      {cm[1][1]:5d}")

# 핵심 지표 요약
from sklearn.metrics import f1_score, precision_score, recall_score
f1  = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
pre = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
print(f"\n핵심 지표 (할 일 클래스 기준)")
print(f"  Precision : {pre:.4f}  (예측한 할 일 중 실제 할 일 비율)")
print(f"  Recall    : {rec:.4f}  (실제 할 일 중 맞게 예측한 비율)")
print(f"  F1        : {f1:.4f}  (종합 성능, 목표 ≥ 0.65)")

if f1 >= 0.75:
    print("\n✅ 좋음 — 실용 수준 달성")
elif f1 >= 0.65:
    print("\n⚠️  보통 — 사용 가능하나 추가 데이터 권장")
else:
    print("\n❌ 미흡 — 재학습 또는 데이터 보강 필요")

## 6. 임계값(Threshold) 분석
0.5가 최적인지 확인합니다. Recall을 높이려면 낮추고, Precision을 높이려면 올립니다.

In [ ]:
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>8} {'F1':>8} {'할일예측수':>10}")
print("-" * 55)

best_f1, best_thresh = 0, 0.5
for thresh in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]:
    preds = [1 if c >= thresh else 0 for c in y_conf]
    p = precision_score(y_true, preds, pos_label=1, zero_division=0)
    r = recall_score(y_true, preds, pos_label=1, zero_division=0)
    f = f1_score(y_true, preds, pos_label=1, zero_division=0)
    n = sum(preds)
    mark = " ← 현재" if thresh == 0.5 else (" ← 최적" if f > best_f1 and thresh != 0.5 else "")
    if f > best_f1:
        best_f1, best_thresh = f, thresh
    print(f"{thresh:>10.2f} {p:>10.4f} {r:>8.4f} {f:>8.4f} {n:>10}  {mark}")

print(f"\n최적 threshold: {best_thresh:.2f}  (F1={best_f1:.4f})")
if best_thresh != 0.5:
    print(f"→ predict.py의 BINARY_THRESHOLD를 {best_thresh}로 변경 권장")

## 7. 오분류 샘플 확인
어떤 문장을 틀렸는지 확인합니다.

In [ ]:
# 실제 할 일인데 노이즈로 예측 (False Negative — 놓친 것)
fn = [(t, c) for t, gt, (pred, c) in zip(texts, y_true, results) if gt==1 and pred==0]
# 실제 노이즈인데 할 일로 예측 (False Positive — 잘못 잡은 것)
fp = [(t, c) for t, gt, (pred, c) in zip(texts, y_true, results) if gt==0 and pred==1]

print(f"[놓친 할 일 — False Negative] {len(fn)}개 (낮은 confidence로 탈락)")
for sent, conf in sorted(fn, key=lambda x: x[1])[:10]:  # confidence 낮은 순
    print(f"  conf={conf:.4f}  {sent[:80]}")

print(f"\n[잘못 잡은 문장 — False Positive] {len(fp)}개 (노이즈인데 할 일로 분류)")
for sent, conf in sorted(fp, key=lambda x: -x[1])[:10]:  # confidence 높은 순
    print(f"  conf={conf:.4f}  {sent[:80]}")

## 8. 실제 가정통신문 테스트
전체 통신문 텍스트를 넣으면 할 일 문장만 뽑아줍니다.

In [ ]:
# predict.py의 split_sentences 로직 인라인
_HEADER_ONLY = re.compile(r'^[^.,!?~]{2,40}(안내|공지|알림|공개수업|상담|학습|행사|일정)\s*$')
_OCR_NOISE   = re.compile(
    r'https?://|^www\.|☎\s*\d'
    r'|^\d{1,2}:\d{2}\s*[~\-–]\s*\d{1,2}:\d{2}'
    r'|^[→←↑↓]+\s*$'
)

def split_sentences(text: str) -> list[str]:
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    out = []
    for line in lines:
        if _HEADER_ONLY.match(line) or _OCR_NOISE.search(line):
            continue
        parts = re.split(
            r'(?<=[.!?])\s+|(?<=다\.)\s+|(?<=요\.)\s+|(?<=니다\.)\s+'
            r'|(?<=까\?)\s+|(?<=요\?)\s+|\s+(?=\d+[.)]\s)|\s+(?=[가-힣]\.\s)',
            line,
        )
        out.extend(parts)
    return [s.strip() for s in out if s.strip() and len(s.strip()) > 3]


def extract_todos(notice_text: str, threshold: float = 0.5) -> list[dict]:
    results = []
    sentences = split_sentences(notice_text)
    if not sentences:
        return []
    preds = predict_batch(sentences)
    for sent, (label, conf) in zip(sentences, preds):
        if conf >= threshold:
            results.append({"text": sent, "confidence": conf})
    return results


# ── 샘플 통신문 테스트 ───────────────────────────────────────────────────────
sample_notice = """학부모님 안녕하십니까?
항상 학교 교육에 관심과 협조를 아끼지 않으시는 학부모님께 감사드립니다.
2026학년도 1학기 학부모 상담 운영에 대해 안내해 드립니다.
1. 상담 기간: 2026년 5월 12일(월) ~ 5월 16일(금)
2. 상담 방법: 담임교사와 1:1 면담 (교실 또는 줌)
3. 신청 방법: 학교종이 앱을 통해 5월 9일(금)까지 신청해 주시기 바랍니다.
상담을 원하지 않으시는 경우에도 학교종이 앱에서 '상담 불필요'를 선택해 주십시오.
준비물: 학생 생활기록부 관련 문의 사항을 미리 메모해 오시면 도움이 됩니다.
자세한 사항은 담임선생님께 문의해 주시기 바랍니다.
2026년 5월 2일 서울갈산초등학교장"""

todos = extract_todos(sample_notice)

print("=" * 60)
print("샘플 통신문 → 할 일 추출 결과")
print("=" * 60)
if todos:
    for i, item in enumerate(todos, 1):
        print(f"\n{i}. [{item['confidence']:.4f}] {item['text']}")
else:
    print("추출된 할 일이 없습니다.")
print(f"\n총 {len(todos)}개 할 일 추출")

## 9. 결과 요약
아래 셀을 실행하면 위 모든 결과를 한 번에 정리해 줍니다.

In [ ]:
print("=" * 60)
print("최종 결과 요약")
print("=" * 60)
print(f"모델        : {MODEL_ID}")
print(f"평가 데이터 : {len(records)}개 문장")
print(f"  True(할일): {true_cnt}개 / False(노이즈): {false_cnt}개")
print()
print(f"[성능 — threshold={BINARY_THRESHOLD}]")
print(f"  Precision : {pre:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1        : {f1:.4f}")
print(f"  FN(놓침)  : {len(fn)}개")
print(f"  FP(오탐)  : {len(fp)}개")
print()
print(f"[임계값 분석]")
print(f"  현재 threshold: 0.5")
print(f"  최적 threshold: {best_thresh:.2f}  (F1={best_f1:.4f})")
print()
if f1 >= 0.75:
    print("판정: ✅ 좋음 — 실용 수준 달성")
elif f1 >= 0.65:
    print("판정: ⚠️  보통 — 추가 데이터 권장")
else:
    print("판정: ❌ 미흡 — 재학습 또는 데이터 보강 필요")